# 🧠 Notebook 4 — Métricas & Gráficos

Parte 4 de 4.

Lê `data/results/results.json` (gerado pelo Notebook 3), agrega entre pacientes e produz as tabelas e gráficos — incluindo a **curva de degradação** (desempenho × número de regiões), que é o resultado central do trabalho.

### O que se agrega e o que não
- **Métricas escalares** (acurácia, F1, sensibilidade, FAR, lead time): agregadas como **média entre pacientes** (cada fold LOSO é um paciente).
- **Sinais brutos / PSD**: nunca se faz média entre pacientes — quando ilustrados, usa-se **um exemplo representativo** (isso fica no Notebook 1).

## 4.1 Imports e config

In [ ]:
import os, re, json, io, warnings, gc, html
import html.parser
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mne
import pywt
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from scipy.signal import butter, sosfiltfilt, iirnotch, filtfilt, welch, resample_poly
from scipy.stats  import kurtosis as sp_kurtosis, skew as sp_skew
from tqdm.auto    import tqdm

mne.set_log_level('WARNING')
warnings.filterwarnings('ignore')
sns.set_theme(style='darkgrid', palette='muted')
print("✅ Imports OK")

In [ ]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ── Diretórios ───────────────────────────────────
ROOT_DIR    = 'data'
CHBMIT_DIR  = os.path.join(ROOT_DIR, 'chb-mit')
SIENA_DIR   = os.path.join(ROOT_DIR, 'siena')
SEIZEIT_DIR = os.path.join(ROOT_DIR, 'seizeit2')
MENDELEY_DIR= os.path.join(ROOT_DIR, 'mendeley')
L1_DIR      = os.path.join(ROOT_DIR, 'level1_signals')
L2_DIR      = os.path.join(ROOT_DIR, 'level2_windows')
FEAT_DIR    = os.path.join(ROOT_DIR, 'features')   # subpastas por nível: features/<level>/
RESULTS_DIR = os.path.join(ROOT_DIR, 'results')
for d in [CHBMIT_DIR, SIENA_DIR, SEIZEIT_DIR, MENDELEY_DIR,
          L1_DIR, L2_DIR, FEAT_DIR, RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Filtragem ──────────────────────────────────
F_HP, F_LP, F_ORDER = 0.5, 40.0, 4
NOTCH = {'CHBMIT': 60.0, 'Siena': 50.0, 'SeizeIT2': 50.0, 'Mendeley': 50.0}

# Reamostragem comum (Siena=512, Mendeley=500, demais=256) → fs único para
# tornar features espectrais comparáveis e permitir juntar datasets.
TARGET_FS = 256

# ── Janelamento ────────────────────────────────
WIN_SEC, OVERLAP = 4, 0.50

# ── Rotulagem ──────────────────────────────────
PRE_SEC, POST_SEC, IGAP_SEC = 10*60, 10*60, 10*60
LBL = dict(interictal=0, preictal=1, ictal=2, postictal=3, unknown=-1)
LBL_NAMES = {v: k for k, v in LBL.items()}

# ── Cap de interictal na extração (controla custo do SeizeIT2) ──────────
MAX_INTER_RATIO = 10   # máx. interictal:eventos por paciente (None desliga)

# ── PACIENTES POR DATASET ──────────────────────────────
PATIENTS = {
    'CHBMIT'  : ['chb01', 'chb02', 'chb03', 'chb04', 'chb05'],
    'Siena'   : ['PN00', 'PN01', 'PN03', 'PN05', 'PN06'],
    'Mendeley': ['p10', 'p11', 'p12', 'p13', 'p14'],
    'SeizeIT2': ['sub-001', 'sub-002', 'sub-003', 'sub-004', 'sub-005'],
}
PILOT = {k: v[0] for k, v in PATIENTS.items()}
SEIZEIT_SESSION = 'ses-01'

# ── NÍVEIS DE REDUÇÃO DE CANAIS = subconjuntos de REGIÕES cerebrais ────────
# Por que regiões e não eletrodos fixos? Os datasets usam montagens diferentes
# (CHB-MIT é bipolar, Siena/Mendeley referencial, SeizeIT2 behind-the-ear), então
# "o mesmo eletrodo" não existe entre todos. Mapear cada canal → região e agregar
# por região (1) funciona em qualquer montagem, (2) dá vetor de tamanho fixo para
# poder juntar datasets, e (3) mantém a interpretabilidade (SHAP por região).
REGIONS = ['frontal', 'temporal', 'central', 'parietal', 'occipital']

LEVELS = {
    'R5': ['frontal', 'temporal', 'central', 'parietal', 'occipital'],  # completo (~19 ch)
    'R3': ['frontal', 'temporal', 'central'],                           # reduzido (~8 ch)
    'R2': ['frontal', 'temporal'],                                      # vestível (~4 ch)
    'R1': ['temporal'],                                                 # ultra-compacto / SeizeIT2
}
# Quais datasets participam de cada nível. SeizeIT2 (behind-the-ear, ~região
# temporal) só entra no nível mínimo R1, que é onde ele faz sentido físico.
LEVEL_DATASETS = {
    'R5': ['CHBMIT', 'Siena', 'Mendeley'],
    'R3': ['CHBMIT', 'Siena', 'Mendeley'],
    'R2': ['CHBMIT', 'Siena', 'Mendeley'],
    'R1': ['CHBMIT', 'Siena', 'Mendeley', 'SeizeIT2'],
}

# ── Mapa eletrodo → região (sistema 10-20; aceita variantes T7/T3 etc.) ────
ELECTRODE_REGION = {}
for _e in ['fp1','fp2','f7','f3','fz','f4','f8','af3','af4']:           ELECTRODE_REGION[_e]='frontal'
for _e in ['t3','t4','t5','t6','t7','t8','p7','p8','ft9','ft10','tp7','tp8']: ELECTRODE_REGION[_e]='temporal'
for _e in ['c3','cz','c4','fc1','fc2','cp1','cp2']:                     ELECTRODE_REGION[_e]='central'
for _e in ['p3','pz','p4','po3','po4']:                                 ELECTRODE_REGION[_e]='parietal'
for _e in ['o1','o2','oz']:                                             ELECTRODE_REGION[_e]='occipital'

print("✅ Configurações OK")
print(f"   fs alvo     : {TARGET_FS} Hz | Janela {WIN_SEC}s overlap {int(OVERLAP*100)}%")
print(f"   Pré/Pós     : {PRE_SEC//60}/{POST_SEC//60} min | cap interictal {MAX_INTER_RATIO}:1")
print(f"   Níveis      : " + " | ".join(f"{k}({len(v)} reg)" for k,v in LEVELS.items()))
for k, v in PATIENTS.items():
    print(f"   {k:9s}: {v}")

## 4.2 Carregar resultados

In [ ]:
with open(os.path.join(RESULTS_DIR,'results.json')) as f:
    records = json.load(f)
df = pd.DataFrame(records)
print(f"✅ {len(df)} registros carregados")
if len(df):
    # expande algumas métricas para colunas planas
    df['acc']        = df['window'].apply(lambda m: m['accuracy'])
    df['f1_macro']   = df['window'].apply(lambda m: m['f1_macro'])
    df['sens_pre']   = df['window'].apply(lambda m: m['rec_preictal'])
    df['acc_bal']    = df['balanceada'].apply(lambda m: m['accuracy'] if m else np.nan)
    df['sens_pre_bal']=df['balanceada'].apply(lambda m: m['rec_preictal'] if m else np.nan)
    df['ev_sens']    = df['event'].apply(lambda m: m['event_sensitivity'])
    df['ev_far']     = df['event'].apply(lambda m: m['event_far_per_hour'])
    df['n_crises']   = df['event'].apply(lambda m: m['n_crises'])
    df['n_pred']     = df['event'].apply(lambda m: m['n_predicted'])
    df['n_leads']    = df['event'].apply(lambda m: len(m['lead_times_s']))
    df['lead_mean_s']= df['event'].apply(lambda m: float(np.mean(m['lead_times_s'])) if m['lead_times_s'] else np.nan)
    display(df[['level','scenario','model','tag','test_subject','acc','f1_macro','sens_pre','ev_sens','ev_far']].head(12))

## 4.3 Agregação por nível × cenário × modelo

In [ ]:
# Agrega entre pacientes (média) por (nível, cenário, modelo)
def agg(group):
    leads = [x for m in group['event'] for x in m['lead_times_s']]
    return pd.Series({
        'acc'        : group['acc'].mean(),
        'f1_macro'   : group['f1_macro'].mean(),
        'sens_pre'   : group['sens_pre'].mean(),
        'acc_bal'    : group['acc_bal'].mean(),
        'sens_pre_bal': group['sens_pre_bal'].mean(),
        'ev_sens'    : group['ev_sens'].mean(),
        'ev_far'     : group['ev_far'].mean(),
        'crises'     : int(group['n_crises'].sum()),
        'previstas'  : int(group['n_pred'].sum()),
        'lead_mean_min': (np.mean(leads)/60.0) if leads else np.nan,
        'n_folds'    : len(group),
    })
summary = df.groupby(['level','scenario','model']).apply(agg).reset_index() if len(df) else pd.DataFrame()
order = {'R5':0,'R3':1,'R2':2,'R1':3}
if len(summary):
    summary['lvl_order'] = summary['level'].map(order)
    summary = summary.sort_values(['lvl_order','scenario','model']).drop(columns='lvl_order')
    print("📊 Resumo agregado (média entre pacientes)")
    display(summary.round(3))
    summary.to_csv(os.path.join(RESULTS_DIR,'summary_by_level_scenario_model.csv'), index=False)

## 4.4 Curva de degradação (desempenho × nível de canais)

O **resultado principal**: quanto se perde ao reduzir de 5 regiões (hospital) para 1 (vestível/SeizeIT2).

In [ ]:
# ── CURVA DE DEGRADAÇÃO: desempenho × nível de canais (resultado principal) ────
# Para cada cenário, plota a métrica em função do nível R5→R1.
if len(summary):
    levels_sorted = [l for l in ['R5','R3','R2','R1'] if l in summary['level'].unique()]
    reg_count = {l: len(LEVELS[l]) for l in levels_sorted}
    metrics_plot = [('ev_sens','Sensibilidade por evento'),
                    ('sens_pre','Sensibilidade pré-ictal (janela)'),
                    ('ev_far','FAR (alarmes falsos/hora)')]
    for scen in ['A_intra','B_cross','C_mix']:
        sub = summary[summary['scenario']==scen]
        if sub.empty: continue
        fig, axes = plt.subplots(1, 3, figsize=(16,4))
        fig.suptitle(f'Curva de degradação — cenário {scen}', fontweight='bold')
        for ax,(mc,title) in zip(axes, metrics_plot):
            for mk in sub['model'].unique():
                s = sub[sub['model']==mk].set_index('level').reindex(levels_sorted)
                xs = [reg_count[l] for l in levels_sorted]
                ax.plot(xs, s[mc].values, 'o-', label=mk.upper())
            ax.set_xlabel('número de regiões (R5→5…R1→1)')
            ax.set_title(title); ax.invert_xaxis()
            if mc!='ev_far': ax.set_ylim(0,1.05)
            ax.legend(fontsize=8)
        plt.tight_layout(); plt.show()
    print("↳ R5=5 regiões (hospital) … R1=1 região (vestível/SeizeIT2). Eixo X invertido: ← menos canais.")

## 4.5 Generalização: mesmo dataset (A) vs misturado (C)

Responde se **misturar datasets no treino** melhora a predição em um dataset novo.

In [ ]:
# ── GENERALIZAÇÃO: A_intra vs C_mix (misturar datasets ajuda?) ───────────
if len(summary):
    fig, axes = plt.subplots(1, 2, figsize=(13,4))
    fig.suptitle('Generalização: treino no mesmo dataset (A) vs misturado (C)', fontweight='bold')
    for ax, mc, title in [(axes[0],'ev_sens','Sensibilidade por evento'),
                          (axes[1],'sens_pre','Sensibilidade pré-ictal')]:
        piv = (summary[summary['scenario'].isin(['A_intra','C_mix'])]
               .groupby(['level','scenario'])[mc].mean().unstack('scenario'))
        piv = piv.reindex([l for l in ['R5','R3','R2','R1'] if l in piv.index])
        piv.plot(kind='bar', ax=ax); ax.set_title(title); ax.set_ylim(0,1.05)
        ax.set_xlabel('nível'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()

## 4.6 Modelos por nível (Realista vs Balanceada)

In [ ]:
# ── Comparativo de modelos por nível (cenário A_intra) — Realista vs Balanceada ─
if len(summary):
    sub = summary[summary['scenario']=='A_intra']
    if not sub.empty:
        levels_sorted = [l for l in ['R5','R3','R2','R1'] if l in sub['level'].unique()]
        fig, axes = plt.subplots(1, len(levels_sorted), figsize=(5*len(levels_sorted),4), squeeze=False)
        fig.suptitle('A_intra — acurácia por modelo (Realista vs Balanceada)', fontweight='bold')
        for ax,(lv) in zip(axes[0], levels_sorted):
            s = sub[sub['level']==lv]; x=np.arange(len(s)); w=0.38
            ax.bar(x-w/2, s['acc'], w, label='Realista', color='steelblue')
            ax.bar(x+w/2, s['acc_bal'], w, label='Balanceada', color='darkorange')
            ax.set_xticks(x); ax.set_xticklabels(s['model'].str.upper()); ax.set_ylim(0,1.05)
            ax.set_title(lv); ax.legend(fontsize=8)
        plt.tight_layout(); plt.show()

## 4.7 Matriz de confusão (melhor modelo, R5/A_intra)

In [ ]:
# ── Matriz de confusão 4×4 (melhor modelo no nível R5, cenário A_intra) ─────
if len(df):
    sel = df[(df['level']=='R5') & (df['scenario']=='A_intra')]
    if not sel.empty:
        best = sel.groupby('model')['ev_sens'].mean().idxmax()
        rows = sel[sel['model']==best]
        cm = np.zeros((4,4), int)
        for m in rows['window']: cm += np.array(m['confusion'])
        fig, ax = plt.subplots(figsize=(6,5))
        im = ax.imshow(cm, cmap='Blues')
        ax.set_xticks(range(4)); ax.set_yticks(range(4))
        ax.set_xticklabels([LBL_NAMES[c] for c in range(4)], rotation=45, ha='right')
        ax.set_yticklabels([LBL_NAMES[c] for c in range(4)])
        ax.set_xlabel('Predito'); ax.set_ylabel('Real')
        ax.set_title(f'Matriz de Confusão — R5/A_intra — {best.upper()}')
        thr = cm.max()/2 if cm.max()>0 else 0
        for i in range(4):
            for j in range(4):
                ax.text(j,i,str(cm[i,j]),ha='center',va='center',
                        color='white' if cm[i,j]>thr else 'black')
        plt.colorbar(im, ax=ax, fraction=0.046); plt.tight_layout(); plt.show()

## 4.8 Sumário final

In [ ]:
print("="*66); print("SUMÁRIO FINAL DO ESTUDO"); print("="*66)
if len(summary):
    print("\n▸ Curva de degradação (cenário C_mix, média dos modelos):")
    cmix = summary[summary['scenario']=='C_mix'].groupby('level')[['ev_sens','ev_far','sens_pre']].mean()
    cmix = cmix.reindex([l for l in ['R5','R3','R2','R1'] if l in cmix.index])
    for lv,row in cmix.iterrows():
        print(f"   {lv} ({len(LEVELS[lv])} reg): ev_sens={row['ev_sens']:.2f}  "
              f"FAR/h={row['ev_far']:.2f}  sens_pre={row['sens_pre']:.2f}")
    if 'R5' in cmix.index and 'R1' in cmix.index:
        dq = cmix.loc['R5','ev_sens'] - cmix.loc['R1','ev_sens']
        print(f"\n   ↳ Perda de sensibilidade R5→R1: {dq:+.2f} "
              f"(hospital → vestível). Esse é o resultado central do TCC.")
    print("\n▸ Próximos passos: interpretabilidade (SHAP por região/banda) no melhor")
    print("   modelo, cruzando regiões importantes com o foco de cada paciente.")
print("\n✅ Análise concluída. CSVs em data/results/.")

---
✅ **Fim do pipeline (4/4).** Próximo passo do TCC: **interpretabilidade** (SHAP por região e banda) sobre o melhor modelo, cruzando as regiões mais importantes com o foco epiléptico de cada paciente (disponível na planilha do Mendeley).